# Costruisci app per la generazione di testo

Fino a ora hai visto nel curriculum che ci sono concetti fondamentali come i prompt e perfino un'intera disciplina chiamata "prompt engineering". Molti strumenti con cui puoi interagire come ChatGPT, Office 365, Microsoft Power Platform e altri, ti supportano usando i prompt per realizzare qualcosa.

Per aggiungere un'esperienza simile a un'app, devi capire concetti come prompt, completamenti e scegliere una libreria con cui lavorare. Questo è esattamente ciò che imparerai in questo capitolo.

## Introduzione

In questo capitolo, farai:

- Imparare la libreria openai e i suoi concetti fondamentali.
- Costruire un'app per la generazione di testo usando openai.
- Capire come usare concetti come prompt, temperatura e token per costruire un'app per la generazione di testo.

## Obiettivi di apprendimento

Alla fine di questa lezione, sarai in grado di:

- Spiegare cos'è un'app per la generazione di testo.
- Costruire un'app per la generazione di testo usando openai.
- Configurare la tua app per usare più o meno token e anche cambiare la temperatura, per un output vario.

## Cos'è un'app per la generazione di testo?

Normalmente quando costruisci un'app ha qualche tipo di interfaccia come la seguente:

- Basata su comandi. Le app console sono app tipiche dove scrivi un comando ed essa svolge un compito. Per esempio, `git` è un'app basata su comandi.
- Interfaccia utente (UI). Alcune app hanno interfacce grafiche (GUI) in cui clicchi pulsanti, inserisci testo, selezioni opzioni e altro.

### Le app console e UI sono limitate

Confrontale con un'app basata su comandi dove scrivi un comando: 

- **È limitata**. Non puoi scrivere qualsiasi comando, solo quelli supportati dall'app.
- **Lingua specifica**. Alcune app supportano molte lingue, ma per default l'app è costruita per una lingua specifica, anche se puoi aggiungere altri supporti linguistici.

### Benefici delle app per la generazione di testo

Quindi come è diversa un'app per la generazione di testo?

In un'app per la generazione di testo, hai più flessibilità, non sei limitato a un set di comandi o a una lingua di input specifica. Invece, puoi usare il linguaggio naturale per interagire con l'app. Un altro vantaggio è che stai già interagendo con una fonte di dati che è stata addestrata su un vasto corpus di informazioni, mentre un'app tradizionale potrebbe essere limitata a ciò che è presente in un database.

### Cosa posso costruire con un'app per la generazione di testo?

Ci sono molte cose che puoi costruire. Per esempio:

- **Un chatbot**. Un chatbot che risponde a domande su argomenti, come la tua azienda e i suoi prodotti potrebbe essere una buona corrispondenza.
- **Assistente**. I LLM sono ottimi per cose come riassumere testi, ottenere insight da testi, produrre testi come curriculum e altro.
- **Assistente al codice**. A seconda del modello linguistico che usi, puoi costruire un assistente al codice che ti aiuta a scrivere codice. Per esempio, puoi usare un prodotto come GitHub Copilot oltre a ChatGPT per aiutarti a scrivere codice.

## Come posso iniziare?

Bene, devi trovare un modo per integrarti con un LLM che di solito coinvolge i seguenti due approcci:

- Usare un'API. Qui costruisci richieste web con il tuo prompt e ricevi testo generato in risposta.
- Usare una libreria. Le librerie aiutano a incapsulare le chiamate API e le rendono più facili da usare.

## Librerie/SDK

Ci sono alcune librerie ben note per lavorare con LLM come:

- **openai**, questa libreria rende facile connettersi al tuo modello e inviare prompt.

Poi ci sono librerie che operano a un livello più alto come:

- **Langchain**. Langchain è molto conosciuta e supporta Python.
- **Semantic Kernel**. Semantic Kernel è una libreria di Microsoft che supporta i linguaggi C#, Python e Java.

## Prima app usando openai

Vediamo come costruire la nostra prima app, quali librerie servono, quanto è richiesto e così via.

### Installa openai

  > [!NOTE] Questo passaggio non è necessario se esegui questo notebook su Codespaces o all'interno di un Devcontainer


Ci sono molte librerie per interagire con OpenAI o Azure OpenAI. È possibile usare numerosi linguaggi di programmazione come C#, Python, JavaScript, Java e altri.  
Abbiamo scelto di usare la libreria Python `openai`, quindi useremo `pip` per installarla.

```bash
pip install openai
```

Se non stai eseguendo questo notebook in Codespaces o in un Dev Container, devi anche installare [Python](https://www.python.org/) sulla tua macchina.

### Crea una risorsa

Nel caso non l'avessi già fatto, devi seguire questi passaggi:

- Crea un account su Azure <https://azure.microsoft.com/free/>.
- Crea una risorsa Azure OpenAI in [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst). Consulta questa guida per come [creare una risorsa](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst).


### Trova la chiave API e l'endpoint

A questo punto devi dire alla libreria `openai` quale chiave API usare. Per trovare la tua chiave API, vai alla sezione "Chiavi e Endpoint" della tua risorsa Azure OpenAI e copia il valore "Key 1".

  ![Chiavi e Endpoint risorsa in Azure Portal](https://learn.microsoft.com/azure/ai-foundry/openai/media/quickstarts/endpoint.png?WT.mc_id=academic-105485-koreyst)

Ora che hai copiato queste informazioni, istruiamo le librerie a usarle.

> [!NOTE]
> Vale la pena separare la tua chiave API dal codice. Puoi farlo usando variabili d'ambiente.
> - Imposta la variabile d'ambiente `AZURE_OPENAI_API_KEY` con la tua chiave API nel tuo file .env. Se hai già completato gli esercizi precedenti di questo corso, sei già pronto.


### Configura Azure

Se stai usando Azure OpenAI, ecco come configurare tutto. L'API Responses è servita dall'**endpoint v1** di Azure OpenAI, quindi puntiamo il client `OpenAI` a `<your-endpoint>/openai/v1/`:

```python
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']
```

Qui stiamo impostando:

- `api_key`, questa è la tua chiave API trovata nel Portale Azure.
- `base_url`, questo è il tuo endpoint Azure OpenAI con `/openai/v1/` aggiunto. Puoi trovare l'endpoint nel Portale Azure accanto alla tua chiave API. Usando l'endpoint v1 non devi più passare un `api_version`.
- `deployment`, questo è il nome della distribuzione del modello che hai creato nel portale Foundry.

> [!NOTE]
> `os.environ` è una mappatura che legge le variabili d'ambiente. Puoi usarlo per leggere variabili d'ambiente come `AZURE_OPENAI_API_KEY` e `AZURE_OPENAI_ENDPOINT`.

## Genera testo

Il modo per generare testo è usare il metodo `responses.create`. Ecco un esempio:

```python
prompt = "Complete the following: Once upon a time there was a"

response = client.responses.create(model=deployment, input=prompt, store=False)
print(response.output_text)
```

Nel codice sopra, creiamo un oggetto risposta e passiamo il modello che vogliamo usare e il prompt. Poi stampiamo il testo generato.

### Risposte in chat

L'API Responses è adatta sia per generazione di testo a singolo turno sia per chatbot multi-turno - basta fornire più messaggi nella lista `input` per costruire una conversazione:

```python
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

response = client.responses.create(model=deployment, input=[{"role": "user", "content": "Hello world"}], store=False)
print(response.output_text)
```

Maggiori dettagli su questa funzionalità in un capitolo successivo.

## Esercizio - la tua prima app per la generazione di testo

Ora che abbiamo imparato come configurare e impostare il servizio Azure OpenAI, è ora di costruire la tua prima app per la generazione di testo. Per costruire la tua app, segui questi passaggi:


1. Crea un ambiente virtuale e installa openai:

  > [!NOTE] Questo passaggio non è necessario se esegui questo notebook su Codespaces o all'interno di un Devcontainer


In [ ]:
# Create virtual environment
! python -m venv venv
# Activate virtual environment
! source venv/bin/activate
# Install openai package
! pip install openai

> [!NOTE]
> Se stai usando Windows, digita `venv\Scripts\activate` invece di `source venv/bin/activate`.

> [!NOTE]
> Trova la tua chiave Azure OpenAI andando su https://portal.azure.com/ e cerca `Open AI`, seleziona la `Open AI resource` e poi seleziona `Keys and Endpoint` e copia il valore di `Key 1`.


1. Crea un file *app.py* e inserisci il seguente codice:


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

# add your completion code
prompt = "Complete the following: Once upon a time there was a"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, store=False)

# print response
print(response.output_text)



    Dovresti vedere un output simile al seguente:

    ```output
     molto infelice _____.

    C'era una volta una sirena molto infelice.
    ```


## Diversi tipi di prompt, per cose diverse

Ora hai visto come generare testo usando un prompt. Hai persino un programma attivo e funzionante che puoi modificare e cambiare per generare diversi tipi di testo. 

I prompt possono essere usati per ogni tipo di compito. Per esempio:

- **Generare un tipo di testo**. Per esempio, puoi generare una poesia, domande per un quiz ecc.
- **Ricercare informazioni**. Puoi usare i prompt per cercare informazioni come nell'esempio 'Cosa significa CORS nello sviluppo web?'.
- **Generare codice**. Puoi usare i prompt per generare codice, per esempio sviluppando una espressione regolare usata per validare email o perché no generare un intero programma, come una web app?  

## Un caso d'uso più pratico: un generatore di ricette

Immagina di avere degli ingredienti a casa e vuoi cucinare qualcosa. Per farlo, ti serve una ricetta. Un modo per trovare ricette è usare un motore di ricerca oppure potresti usare un LLM per farlo.

Potresti scrivere un prompt così:

> "Mostrami 5 ricette per un piatto con i seguenti ingredienti: pollo, patate e carote. Per ogni ricetta, elenca tutti gli ingredienti usati"

Dato il prompt sopra, potresti ricevere una risposta simile a:

```output
1. Roasted Chicken and Vegetables: 
Ingredients: 
- 4 chicken thighs
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 2 tablespoons olive oil
- 2 cloves garlic, minced
- 1 teaspoon dried thyme
- 1 teaspoon dried oregano
- Salt and pepper, to taste

2. Chicken and Potato Stew: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 cloves garlic, minced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

3. Chicken and Potato Bake: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

4. Chicken and Potato Soup: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 cloves garlic, minced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 4 cups chicken broth
- Salt and pepper, to taste

5. Chicken and Potato Hash: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
```

Questo risultato è ottimo, so cosa cucinare. A questo punto, delle migliorie utili potrebbero essere:

- Filtrare gli ingredienti che non mi piacciono o a cui sono allergico.
- Produrre una lista della spesa, nel caso non abbia tutti gli ingredienti in casa.

Per i casi sopra, aggiungiamo un prompt aggiuntivo:

> "Per favore rimuovi le ricette con aglio perché sono allergico e sostituiscilo con qualcos'altro. Inoltre, per favore produci una lista della spesa per le ricette, considerando che ho già pollo, patate e carote in casa."

Ora hai un nuovo risultato, ovvero:

```output
1. Roasted Chicken and Vegetables: 
Ingredients: 
- 4 chicken thighs
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 2 tablespoons olive oil
- 1 teaspoon dried thyme
- 1 teaspoon dried oregano
- Salt and pepper, to taste

2. Chicken and Potato Stew: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

3. Chicken and Potato Bake: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

4. Chicken and Potato Soup: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 4 cups chicken broth
- Salt and pepper, to taste

5. Chicken and Potato Hash: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 1 teaspoon dried oregano

Shopping List: 
- Olive oil
- Onion
- Thyme
- Oregano
- Salt
- Pepper
```

Queste sono le tue cinque ricette, senza aglio menzionato e hai anche una lista della spesa considerando quello che hai già in casa. 


## Esercizio - crea un generatore di ricette

Ora che abbiamo messo in scena uno scenario, scriviamo del codice che corrisponda allo scenario dimostrato. Per farlo, segui questi passaggi:

1. Usa il file *app.py* esistente come punto di partenza
1. Individua la variabile `prompt` e modifica il suo codice con il seguente:


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

prompt = "Show me 5 recipes for a dish with the following ingredients: chicken, potatoes, and carrots. Per recipe, list all the ingredients used"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, max_output_tokens=600, store=False)

# print response
print(response.output_text)


Se ora esegui il codice, dovresti vedere un output simile a:

```output
-Chicken Stew with Potatoes and Carrots: 3 tablespoons oil, 1 onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 bay leaf, 1 thyme sprig, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 1 1/2 cups chicken broth, 1/2 cup dry white wine, 2 tablespoons chopped fresh parsley, 2 tablespoons unsalted butter, 1 1/2 pounds boneless, skinless chicken thighs, cut into 1-inch pieces
-Oven-Roasted Chicken with Potatoes and Carrots: 3 tablespoons extra-virgin olive oil, 1 tablespoon Dijon mustard, 1 tablespoon chopped fresh rosemary, 1 tablespoon chopped fresh thyme, 4 cloves garlic, minced, 1 1/2 pounds small red potatoes, quartered, 1 1/2 pounds carrots, quartered lengthwise, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 1 (4-pound) whole chicken
-Chicken, Potato, and Carrot Casserole: cooking spray, 1 large onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and shredded, 1 potato, peeled and shredded, 1/2 teaspoon dried thyme leaves, 1/4 teaspoon salt, 1/4 teaspoon black pepper, 2 cups fat-free, low-sodium chicken broth, 1 cup frozen peas, 1/4 cup all-purpose flour, 1 cup 2% reduced-fat milk, 1/4 cup grated Parmesan cheese

-One Pot Chicken and Potato Dinner: 2 tablespoons olive oil, 1 pound boneless, skinless chicken thighs, cut into 1-inch pieces, 1 large onion, chopped, 3 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 bay leaf, 1 thyme sprig, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 2 cups chicken broth, 1/2 cup dry white wine

-Chicken, Potato, and Carrot Curry: 1 tablespoon vegetable oil, 1 large onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 teaspoon ground coriander, 1 teaspoon ground cumin, 1/2 teaspoon ground turmeric, 1/2 teaspoon ground ginger, 1/4 teaspoon cayenne pepper, 2 cups chicken broth, 1/2 cup dry white wine, 1 (15-ounce) can chickpeas, drained and rinsed, 1/2 cup raisins, 1/2 cup chopped fresh cilantro
```

> NOTA, il tuo LLM è nondeterministico, quindi potresti ottenere risultati diversi ogni volta che esegui il programma.

Ottimo, vediamo come possiamo migliorare le cose. Per migliorare le cose, vogliamo assicurarci che il codice sia flessibile, così ingredienti e numero di ricette possono essere migliorati e modificati. 


1. Cambiamo il codice nel modo seguente:


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

no_recipes = input("No of recipes (for example, 5: ")

ingredients = input("List of ingredients (for example, chicken, potatoes, and carrots: ")

# interpolate the number of recipes into the prompt an ingredients
prompt = f"Show me {no_recipes} recipes for a dish with the following ingredients: {ingredients}. Per recipe, list all the ingredients used"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, max_output_tokens=600, store=False)

# print response
print(response.output_text)


-Torta alla fragola: latte, farina, lievito in polvere, zucchero, sale, burro non salato, fragole, panna montata        
-Latte alla fragola: latte, fragole, zucchero, estratto di vaniglia
```

### Improve by adding filter and shopping list

We now have a working app capable of producing recipes and it's flexible as it relies on inputs from the user, both on the number of recipes but also the ingredients used.

To further improve it, we want to add the following:

- **Filter out ingredients**. We want to be able to filter out ingredients we don't like or are allergic to. To accomplish this change, we can edit our existing prompt and add a filter condition to the end of it like so:

    ```python
    filter = input("Filter (for example, vegetarian, vegan, or gluten-free: ")

    prompt = f"Show me {no_recipes} recipes for a dish with the following ingredients: {ingredients}. Per recipe, list all the ingredients used, no {filter}"
    ```

    Above, we add `{filter}` to the end of the prompt and we also capture the filter value from the user.

    An example input of running the program can now look like so:
    
    ```output    
    No of recipes (for example, 5: 3
    List of ingredients (for example, chicken, potatoes, and carrots: onion,milk
    Filter (for example, vegetarian, vegan, or gluten-free: no milk

    1. French Onion Soup

    Ingredients:
    
    -1 large onion, sliced
    -3 cups beef broth
    -1 cup milk
    -6 slices french bread
    -1/4 cup shredded Parmesan cheese
    -1 tablespoon butter
    -1 teaspoon dried thyme
    -1/4 teaspoon salt
    -1/4 teaspoon black pepper
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add beef broth, milk, thyme, salt, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. Place french bread slices on soup bowls.
    5. Ladle soup over bread.
    6. Sprinkle with Parmesan cheese.
    
    2. Onion and Potato Soup
    
    Ingredients:
    
    -1 large onion, chopped
    -2 cups potatoes, diced
    -3 cups vegetable broth
    -1 cup milk
    -1/4 teaspoon black pepper
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add potatoes, vegetable broth, milk, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. Serve hot.
    
    3. Creamy Onion Soup
    
    Ingredients:
    
    -1 large onion, chopped
    -3 cups vegetable broth
    -1 cup milk
    -1/4 teaspoon black pepper
    -1/4 cup all-purpose flour
    -1/2 cup shredded Parmesan cheese
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add vegetable broth, milk, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. In a small bowl, whisk together flour and Parmesan cheese until smooth.
    5. Add to soup and simmer for an additional 5 minutes, or until soup has thickened.
    ```

    As you can see, any recipes with milk in it has been filtered out. But, if you're lactose intolerant, you might want to filter out recipes with cheese in them as well, so there's a need to be clear.

- **Produce a shopping list**. We want to produce a shopping list, considering what we already have at home.

    For this functionality, we could either try to solve everything in one prompt or we could split it up into two prompts. Let's try the latter approach. Here we're suggesting adding an additional prompt, but for that to work, we need to add the result of the former prompt as context to the latter prompt. 

    Locate the part in the code that prints out the result from the first prompt and add the following code below:
    
    ```python
    old_prompt_result = response.output_text
    prompt = "Produce a shopping list for the generated recipes and please don't include ingredients that I already have."
    
    new_prompt = f"{old_prompt_result} {prompt}"
    messages = [{"role": "user", "content": new_prompt}]
    response = client.responses.create(model=deployment, input=messages, max_output_tokens=1200, store=False)
    
    # print response
    print("Shopping list:")
    print(response.output_text)
    ```

    Note the following:

    - We're constructing a new prompt by adding the result from the first prompt to the new prompt: 
    
        ```python
        new_prompt = f"{old_prompt_result} {prompt}"
        messages = [{"role": "user", "content": new_prompt}]
        ```

    - We make a new request, but also considering the number of tokens we asked for in the first prompt, so this time we say `max_output_tokens` is 1200. 

        ```python
        response = client.responses.create(model=deployment, input=messages, max_output_tokens=1200, store=False)
        ```  

        Taking this code for a spin, we now arrive at the following output:

        ```output
        No of recipes (for example, 5: 2
        List of ingredients (for example, chicken, potatoes, and carrots: apple,flour
        Filter (for example, vegetarian, vegan, or gluten-free: sugar
        Recipes:
         or milk.
        
        -Apple and flour pancakes: 1 cup flour, 1/2 tsp baking powder, 1/2 tsp baking soda, 1/4 tsp salt, 1 tbsp sugar, 1 egg, 1 cup buttermilk or sour milk, 1/4 cup melted butter, 1 Granny Smith apple, peeled and grated
        -Apple fritters: 1-1/2 cups flour, 1 tsp baking powder, 1/4 tsp salt, 1/4 tsp baking soda, 1/4 tsp nutmeg, 1/4 tsp cinnamon, 1/4 tsp allspice, 1/4 cup sugar, 1/4 cup vegetable shortening, 1/4 cup milk, 1 egg, 2 cups shredded, peeled apples
        Shopping list:
         -Flour, baking powder, baking soda, salt, sugar, egg, buttermilk, butter, apple, nutmeg, cinnamon, allspice 
        ```
        
- **A word on token length**. We should consider how many tokens we need to generate the text we want. Tokens cost money, so where possible, we should try to be economical with the number of tokens we use. For example, can we phrase the prompt so that we can use less tokens?

   To change tokens used, you can use the `max_output_tokens` parameter. For example, if you want to use 100 tokens, you would do:

    ```python
    response = client.responses.create(model=deployment, input=messages, max_output_tokens=100, store=False)
    ```

- **Experimenting with temperature**. Temperature is something we haven't mentioned so far but is an important context for how our program performs. The higher the temperature value the more random the output will be. Conversely the lower the temperature value the more predictable the output will be. Consider whether you want variation in your output or not.

   To alter the temperature, you can use the `temperature` parameter. For example, if you want to use a temperature of 0.5, you would do:

    ```python
    response = client.responses.create(model=deployment, input=messages, store=False)
    ```

   > Note, the closer to 1.0, the more varied the output.



## Assignment

For this assignment, you can choose what to build.

Here are some suggestions:

- Tweak the recipe generator app to improve it further. Play around with temperature values, and the prompts to see what you can come up with.
- Build a "study buddy". This app should be able to answer questions about a topic for example Python, you could have prompts like "What is a certain topic in Python?", or you could have a prompt that says, show me code for a certain topic etc.
- History bot, make history come alive, instruct the bot to play a certain historical character and ask it questions about its life and times. 

## Solution

### Study buddy

- "You're an expert on the Python language

    Suggest a beginner lesson for Python in the following format:
    
    Format:
    - concepts:
    - brief explanation of the lesson:
    - exercise in code with solutions"

Above is a starter prompt, see how you can use it and tweak it to your liking.

### History bot

Here's some prompts you could be using:

- "You are Abe Lincoln, tell me about yourself in 3 sentences, and respond using grammar and words like Abe would have used"
- "You are Abe Lincoln, respond using grammar and words like Abe would have used:

   Tell me about your greatest accomplishments, in 300 words:"

## Knowledge check

What does the concept temperature do?

1. It controls how random the output is.
1. It controls how big the response is.
1. It controls how many tokens are used.

A: 1

What's a good way to store secrets like API keys?

1. In code.
1. In a file.
1. In environment variables.

A: 3, because environment variables are not stored in code and can be loaded from the code. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
